<a href="https://colab.research.google.com/github/win-eva/als-sex-stratified-target-discovery/blob/main/03_phenotypic_rescue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import pandas as pd
from pathlib import Path

drive.mount("/content/drive")

female_path = Path("/content/drive/MyDrive/ALS Data/female_CLUE_results")
male_path = Path("/content/drive/MyDrive/ALS Data/male_CLUE_results")

## CLUE reversal results

CLUE returns its normalised connectivity scores as a .gct file (a GSEA-format matrix
with a couple of header rows before the actual table).

In [ ]:
def read_gct(path):
    with open(path, "r") as f:
        f.readline()  # version
        f.readline()  # dimensions
    return pd.read_csv(path, sep="\t", skiprows=2)

female_ncs = read_gct(female_path / "ncs.gct")
male_ncs = read_gct(male_path / "ncs.gct")

# drop the metadata row, keep only real compound/perturbagen signatures
female = female_ncs[female_ncs["id"] != "desc"].copy()
male = male_ncs[male_ncs["id"] != "desc"].copy()

female["TAG"] = pd.to_numeric(female["TAG"], errors="coerce")
male["TAG"] = pd.to_numeric(male["TAG"], errors="coerce")
female = female.dropna(subset=["TAG"])
male = male.dropna(subset=["TAG"])

## Highest 2,000 reversing compounds per sex

CLUE also returns genetic perturbations (knockdowns, overexpression) alongside small
molecules -> only the compound signatures (`trt_cp`) go forward. Signatures are
collapsed to one row per compound (a compound can appear in many cell lines / doses /
timepoints) and ranked by mean TAG; more negative TAG means stronger reversal of the
disease signature.

In [ ]:
female_cp = female[female["pert_type"] == "trt_cp"].copy()
male_cp = male[male["pert_type"] == "trt_cp"].copy()

def top_compounds(df, top_n=2000):
    ranked = (
        df.groupby("pert_iname")
        .agg(mean_TAG=("TAG", "mean"), best_TAG=("TAG", "min"), n_sig=("TAG", "count"))
        .reset_index()
        .sort_values("mean_TAG")
    )
    return ranked.head(top_n)

female_top2000 = top_compounds(female_cp)
male_top2000 = top_compounds(male_cp)

print("Female:", len(female_cp), "compound signatures ->", female_cp["pert_iname"].nunique(), "unique compounds")
print("Male:", len(male_cp), "compound signatures ->", male_cp["pert_iname"].nunique(), "unique compounds")

## ChEMBL target annotation

In [ ]:
!pip install chembl_webresource_client -q

In [ ]:
import re
import time
from chembl_webresource_client.new_client import new_client

BASE = Path("/content/drive/MyDrive/ALS Data")
COMPOUNDINFO = BASE / "compoundinfo_beta.txt"

MIN_N_SIG = 5          # minimum CLUE signatures per compound to be considered reliable
MAX_IC50_NM = 10000     # 10 uM binding affinity cutoff for a ChEMBL activity to count as a hit
MIN_INHIBITION = 50     # % inhibition threshold for that activity type
SLEEP_SEC = 0.25        # stay under the ChEMBL API rate limit

Compounds investigated in ALS clinical trials or used for symptom management were
looked up manually and their known targets pulled from ChEMBL (see cell below). Genes
in this set are labelled disease-modifying; everything else novel.

In [ ]:
ALS_TRIAL_AND_SYMPTOMATIC_TARGETS = {
    # riluzole / sodium channel modulators
    "SCN4A", "SCN9A", "SCN8A", "SCN1A", "SCN2A", "SCN5A", "SCN10A",
    # tofersen / edaravone
    "SOD1", "SOD2",
    # phenylbutyrate / valproate / lithium (HDAC + GSK3B axis)
    "HDAC1", "HDAC2", "HDAC3", "HDAC4", "HDAC5", "HDAC6",
    "HDAC7", "HDAC8", "HDAC9", "HDAC10", "HDAC11", "GSK3B", "GSK3A",
    # TUDCA
    "NR1H4", "SLC10A1", "PLA2G1B",
    # rapamycin
    "MTOR", "FKBP1A", "FKBP12",
    # tamoxifen
    "ESR1", "ESR2", "ABCB1",
    # gabapentin
    "CACNA2D1", "CACNA2D2",
    # memantine
    "GRIN1", "GRIN2A", "GRIN2B", "GRIN2C", "GRIN2D",
    # celecoxib
    "PTGS1", "PTGS2",
    # pioglitazone
    "PPARG", "PPARA", "PPARD", "CISD1", "MAOB",
    # arimoclomol
    "HSF1", "HSPA1A", "HSPA1B", "HSP90AA1", "HSP90AB1",
    # tirasemtiv / reldesemtiv
    "TNNI1", "TNNT3", "TNNC2",
    # symptomatic: spasticity, mood, autonomic symptom management
    "GABBR1", "GABBR2", "GABRA1", "GABRA2", "GABRA3", "GABRA5", "GABRB1",
    "ADRA2A", "ADRA2B", "ADRA2C", "SLC6A4", "SLC6A2", "SLC6A3",
    "HRH1", "HRH2", "CHRM1", "CHRM2", "CHRM3", "CHRM4", "CHRM5",
    "SIGMAR1", "OPRM1", "OPRK1", "OPRD1", "DRD2", "DRD3", "DRD4",
    "HTR1A", "HTR2A", "HTR2B", "HTR2C", "HTR3A", "KCNH2",
}

def annotate_tier(gene):
    return "disease-modifying" if gene in ALS_TRIAL_AND_SYMPTOMATIC_TARGETS else "novel"

In [ ]:
def clean_name(x):
    return re.sub(r"[^a-z0-9 ]", " ", str(x).lower()).strip()

def is_brd(name):
    return name.upper().startswith("BRD") or name.upper().startswith("VU-")

def load_compoundinfo(path):
    ci = pd.read_csv(path, sep="\t", low_memory=False)
    print(f"compoundinfo: {len(ci)} compounds ({ci['inchi_key'].notna().sum()} with InChIKey, {ci['canonical_smiles'].notna().sum()} with SMILES)")
    return ci

ci = load_compoundinfo(COMPOUNDINFO)

In [ ]:
def prepare_top_compounds(df_cp, top_n=2000, min_n_sig=MIN_N_SIG):
    def clean_join(values):
        vals = set(map(str, values.dropna())) - {"-666", "nan", ""}
        return "; ".join(sorted(vals))

    def target_join(values):
        targets = set()
        for v in values:
            v = str(v).strip()
            if v in {"-666", "nan", "", "None"}:
                continue
            targets.update(t.strip().upper() for t in v.split("|") if t.strip())
        return "|".join(sorted(targets))

    top = (
        df_cp.groupby("pert_iname")
        .agg(
            mean_TAG=("TAG", "mean"),
            best_TAG=("TAG", "min"),
            n_sig=("TAG", "count"),
            clue_moa=("moa", clean_join),
            clue_target=("target_name", target_join),
        )
        .reset_index()
        .query("n_sig >= @min_n_sig")
        .sort_values("mean_TAG")
        .head(top_n)
        .reset_index(drop=True)
    )
    top["rank"] = top.index + 1
    top["drug_clean"] = top["pert_iname"].apply(clean_name)
    top["source_type"] = top["pert_iname"].apply(lambda x: "brd_vu" if is_brd(x) else "named")

    # bring in InChIKey / SMILES / compoundinfo's own target+moa annotations
    ci_meta = (
        ci[ci["pert_id"].isin(top["pert_iname"])]
        .rename(columns={"pert_id": "pert_iname"})
        .groupby("pert_iname")
        .agg(
            ci_inchikey=("inchi_key", "first"),
            ci_smiles=("canonical_smiles", "first"),
            ci_target=("target", lambda x: "|".join(str(v).upper() for v in x.dropna() if str(v) not in {"-666", "nan", ""})),
            ci_moa=("moa", lambda x: "; ".join(str(v) for v in x.dropna() if str(v) not in {"-666", "nan", ""})),
        )
        .reset_index()
    )
    return top.merge(ci_meta, on="pert_iname", how="left")

In [ ]:
def chembl_by_name(name):
    mol = new_client.molecule
    r = mol.filter(pref_name__iexact=name).only(["molecule_chembl_id", "molecule_type"])
    if r:
        return r[0]["molecule_chembl_id"], r[0].get("molecule_type", "")
    r = mol.filter(molecule_synonyms__molecule_synonym__iexact=name).only(["molecule_chembl_id", "molecule_type"])
    if r:
        return r[0]["molecule_chembl_id"], r[0].get("molecule_type", "")
    return None, None

def chembl_by_inchikey(inchi_key):
    mol = new_client.molecule
    r = mol.filter(molecule_structures__standard_inchi_key=inchi_key).only(["molecule_chembl_id", "molecule_type"])
    return (r[0]["molecule_chembl_id"], r[0].get("molecule_type", "")) if r else (None, None)

def chembl_by_smiles(smiles):
    try:
        mol = new_client.molecule
        r = mol.filter(molecule_structures__canonical_smiles=smiles).only(["molecule_chembl_id", "molecule_type"])
        return (r[0]["molecule_chembl_id"], r[0].get("molecule_type", "")) if r else (None, None)
    except Exception:
        return None, None

def chembl_get_gene(target_chembl_id):
    if not target_chembl_id:
        return ""
    try:
        result = new_client.target.filter(target_chembl_id=target_chembl_id).only(["target_components"])
        if not result:
            return ""
        for comp in result[0].get("target_components", []):
            for syn in comp.get("target_component_synonyms", []):
                if syn.get("syn_type") == "GENE_SYMBOL":
                    return syn["component_synonym"]
        return ""
    except Exception:
        return ""

def chembl_get_targets(chembl_id):
    """Binding activities for a compound, kept if they clear the affinity/inhibition
    cutoff; mean pChEMBL is computed later, across everything returned here."""
    acts = new_client.activity.filter(
        molecule_chembl_id=chembl_id, assay_type="B"
    ).only(["target_chembl_id", "target_pref_name", "standard_type", "standard_value", "standard_units", "pchembl_value"])

    results, seen = [], set()
    for a in acts:
        target = a.get("target_pref_name", "")
        if not target or target in seen:
            continue
        val, std_type, units = a.get("standard_value"), a.get("standard_type", ""), a.get("standard_units", "")
        passes = False
        if val is not None:
            try:
                val_f = float(val)
                if std_type in ("IC50", "Ki", "Kd", "EC50", "AC50") and units == "nM":
                    passes = val_f <= MAX_IC50_NM
                elif std_type == "Inhibition":
                    passes = val_f >= MIN_INHIBITION
                elif std_type in ("pIC50", "pKi"):
                    passes = val_f >= 5
            except (ValueError, TypeError):
                pass
        if not passes:
            continue
        seen.add(target)
        results.append({
            "chembl_target_id": a.get("target_chembl_id", ""),
            "target_name": target,
            "pchembl": a.get("pchembl_value"),
        })
    return results

**Two annotation sources**: CLUE (its own `target_name` field
plus compoundinfo.txt's `target` column) and ChEMBL (actual binding assay lookups). `n_sources` below can
only be 1 or 2 as a result.

In [ ]:
def run_annotation_pipeline(clue_cp, sex, top_n=2000):
    print(f"\n{sex.upper()} -- top {top_n} compounds")
    top = prepare_top_compounds(clue_cp, top_n=top_n)
    print(f"  Named: {(top['source_type']=='named').sum()} | BRD/VU: {(top['source_type']=='brd_vu').sum()}")

    detail_rows = []

    # ChEMBL: named compounds by name (falling back to InChIKey if that fails)
    named = top[top["source_type"] == "named"]
    chembl_hits = 0
    for _, row in named.iterrows():
        cid, mol_type = chembl_by_name(row["drug_clean"])
        time.sleep(SLEEP_SEC)
        if cid is None and pd.notna(row.get("ci_inchikey")):
            cid, mol_type = chembl_by_inchikey(row["ci_inchikey"])
            time.sleep(SLEEP_SEC)
        if cid is None:
            continue
        chembl_hits += 1
        for tgt in chembl_get_targets(cid):
            gene = chembl_get_gene(tgt["chembl_target_id"])
            time.sleep(SLEEP_SEC * 0.5)
            if gene:
                detail_rows.append({"pert_iname": row["pert_iname"], "rank": row["rank"],
                                     "mean_TAG": row["mean_TAG"], "target_gene": gene.upper(),
                                     "pchembl": tgt["pchembl"], "source": "chembl"})
    print(f"  ChEMBL hits (named): {chembl_hits} / {len(named)}")

    # ChEMBL: BRD/VU compounds via InChIKey, falling back to SMILES
    brd = top[top["source_type"] == "brd_vu"]
    brd_inchi = brd[brd["ci_inchikey"].notna()]
    brd_smiles_only = brd[brd["ci_inchikey"].isna() & brd["ci_smiles"].notna()]

    inchi_hits = 0
    for _, row in brd_inchi.iterrows():
        cid, mol_type = chembl_by_inchikey(row["ci_inchikey"])
        time.sleep(SLEEP_SEC)
        if cid is None:
            continue
        inchi_hits += 1
        for tgt in chembl_get_targets(cid):
            gene = chembl_get_gene(tgt["chembl_target_id"])
            time.sleep(SLEEP_SEC * 0.5)
            if gene:
                detail_rows.append({"pert_iname": row["pert_iname"], "rank": row["rank"],
                                     "mean_TAG": row["mean_TAG"], "target_gene": gene.upper(),
                                     "pchembl": tgt["pchembl"], "source": "chembl"})
    print(f"  ChEMBL hits (BRD via InChIKey): {inchi_hits} / {len(brd_inchi)}")

    smiles_hits = 0
    for _, row in brd_smiles_only.iterrows():
        cid, mol_type = chembl_by_smiles(row["ci_smiles"])
        time.sleep(SLEEP_SEC)
        if cid is None:
            continue
        smiles_hits += 1
        for tgt in chembl_get_targets(cid):
            gene = chembl_get_gene(tgt["chembl_target_id"])
            time.sleep(SLEEP_SEC * 0.5)
            if gene:
                detail_rows.append({"pert_iname": row["pert_iname"], "rank": row["rank"],
                                     "mean_TAG": row["mean_TAG"], "target_gene": gene.upper(),
                                     "pchembl": tgt["pchembl"], "source": "chembl"})
    print(f"  ChEMBL hits (BRD via SMILES): {smiles_hits} / {len(brd_smiles_only)}")

    # unresolvable BRD compounds (no InChIKey or SMILES in compoundinfo) are excluded here

    # CLUE: its own target annotations, plus compoundinfo.txt's target column
    clue_annot = 0
    for _, row in top.iterrows():
        genes = set()
        for field in ("clue_target", "ci_target"):
            val = str(row.get(field, "")).strip()
            if val and val not in {"nan", ""}:
                genes.update(g.strip().upper() for g in val.split("|") if g.strip() and g.strip() not in {"-666", "NAN"})
        if not genes:
            continue
        clue_annot += 1
        for gene in genes:
            detail_rows.append({"pert_iname": row["pert_iname"], "rank": row["rank"],
                                 "mean_TAG": row["mean_TAG"], "target_gene": gene,
                                 "pchembl": None, "source": "clue"})
    print(f"  CLUE/compoundinfo annotated compounds: {clue_annot}")

    detail_df = pd.DataFrame(detail_rows)
    detail_df["target_gene"] = detail_df["target_gene"].str.strip().str.upper()
    detail_df = detail_df[detail_df["target_gene"] != ""]

    n_sources = (
        detail_df.groupby(["pert_iname", "target_gene"])["source"]
        .nunique().reset_index().rename(columns={"source": "n_sources"})
    )
    detail_df = detail_df.merge(n_sources, on=["pert_iname", "target_gene"], how="left")

    def safe_mean(x):
        vals = pd.to_numeric(x, errors="coerce").dropna()
        return round(vals.mean(), 3) if len(vals) else None

    summary = (
        detail_df.groupby("target_gene")
        .agg(
            n_compounds=("pert_iname", "nunique"),
            reversal_score=("mean_TAG", lambda x: round(abs(x).sum(), 4)),
            mean_pchembl=("pchembl", safe_mean),
            sources=("source", lambda x: "; ".join(sorted(set(x)))),
            n_sources=("n_sources", "max"),
        )
        .reset_index()
        .sort_values("reversal_score", ascending=False)
        .reset_index(drop=True)
    )
    summary["pathway_tier"] = summary["target_gene"].apply(annotate_tier)
    summary.index += 1
    summary.index.name = "rank"

    n_novel = (summary["pathway_tier"] == "novel").sum()
    n_dm = (summary["pathway_tier"] == "disease-modifying").sum()
    print(f"  {sex}: {n_novel} novel targets, {n_dm} disease-modifying targets")

    detail_df.to_csv(BASE / f"als_detail_{sex}.csv", index=False)
    summary.to_csv(BASE / f"als_targets_{sex}.csv")
    return detail_df, summary

In [ ]:
female_detail, female_targets = run_annotation_pipeline(female_cp, "female")
male_detail, male_targets = run_annotation_pipeline(male_cp, "male")